# 04 — Backward Op

## Background

Deep learning frameworks generate gradient kernels automatically during backpropagation. However, for custom operators or performance-critical paths, hand-written gradient kernels in Triton or TileLang are essential.

This notebook uses **broadcast multiply + ReLU** as a case study, demonstrating how to implement both forward and backward passes across three frameworks.

**Broadcasting**: weight vector `B [M]` is broadcast along the N dimension, multiplied element-wise with feature matrix `A [N, M]` — a pattern common in output scaling and normalisation.

## Problem Definition

**Forward (Broadcast Mul + ReLU)**:
$$C[i, j] = \max\bigl(0,\ A[i, j] \times B[j]\bigr)$$

**Backward (chain rule, gradient w.r.t. A)**:
$$\frac{\partial\mathcal{L}}{\partial A[i,j]} = \frac{\partial\mathcal{L}}{\partial C[i,j]} \times B[j] \times \mathbf{1}[A[i,j] \times B[j] > 0]$$

where $\mathbf{1}[\cdot]$ is the subgradient (ReLU gate).

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
N, M = 8192, 4096
A  = torch.randn(N, M, dtype=torch.float16, device="cuda")
B  = torch.randn(M,    dtype=torch.float16, device="cuda")
dC = torch.randn(N, M, dtype=torch.float16, device="cuda")

def ref_fwd(A, B):       return torch.relu(A * B)
def ref_bwd(A, B, dC):
    A_ = A.clone().float().requires_grad_(True)
    torch.relu(A_ * B.float()).backward(dC.float())
    return A_.grad.half()

C_ref  = ref_fwd(A, B)
dA_ref = ref_bwd(A, B, dC)
print(f"Forward:  {A.shape} × {B.shape} → {C_ref.shape}")
print(f"Backward: {dC.shape} → dA {dA_ref.shape}")

## Triton Implementation

Forward: 2D grid, each block handles a `[BN, BM]` tile; `B[j]` is broadcast along rows.  
Backward: same grid; the ReLU gate `(A*B > 0).to(dtype)` implements the subgradient.

In [ ]:
@triton.jit
def _fwd_kernel(A_ptr, B_ptr, C_ptr, N, M, BN: tl.constexpr, BM: tl.constexpr):
    pid_n = tl.program_id(0);  pid_m = tl.program_id(1)
    rows = pid_n * BN + tl.arange(0, BN)
    cols = pid_m * BM + tl.arange(0, BM)
    mask = (rows[:, None] < N) & (cols[None, :] < M)
    a = tl.load(A_ptr + rows[:, None] * M + cols[None, :], mask=mask, other=0.0)
    b = tl.load(B_ptr + cols, mask=cols < M, other=0.0)
    tl.store(C_ptr + rows[:, None] * M + cols[None, :], tl.maximum(a * b[None, :], 0.0), mask=mask)

@triton.jit
def _bwd_kernel(A_ptr, B_ptr, dC_ptr, dA_ptr, N, M, BN: tl.constexpr, BM: tl.constexpr):
    pid_n = tl.program_id(0);  pid_m = tl.program_id(1)
    rows = pid_n * BN + tl.arange(0, BN)
    cols = pid_m * BM + tl.arange(0, BM)
    mask = (rows[:, None] < N) & (cols[None, :] < M)
    a  = tl.load(A_ptr  + rows[:, None] * M + cols[None, :], mask=mask, other=0.0)
    b  = tl.load(B_ptr  + cols, mask=cols < M, other=0.0)
    dc = tl.load(dC_ptr + rows[:, None] * M + cols[None, :], mask=mask, other=0.0)
    gate = (a * b[None, :] > 0).to(a.dtype)
    tl.store(dA_ptr + rows[:, None] * M + cols[None, :], dc * b[None, :] * gate, mask=mask)

BN_t, BM_t = 64, 64

def triton_fwd(A, B):
    C = torch.empty_like(A)
    grid = (triton.cdiv(N, BN_t), triton.cdiv(M, BM_t))
    _fwd_kernel[grid](A, B, C, N, M, BN=BN_t, BM=BM_t)
    return C

def triton_bwd(A, B, dC):
    dA = torch.empty_like(A)
    grid = (triton.cdiv(N, BN_t), triton.cdiv(M, BM_t))
    _bwd_kernel[grid](A, B, dC, dA, N, M, BN=BN_t, BM=BM_t)
    return dA

ok_fwd = torch.allclose(C_ref,  triton_fwd(A, B),       atol=1e-2)
# float16 boundary: subnormal A*B products round to 0 in fp16 but not fp32.
# Use fp16-consistent reference (dC * B * (A*B>0) computed in fp16) for the bwd check.
dA_ref_f16 = dC * B * (A * B > 0).to(torch.float16)
ok_bwd = torch.allclose(dA_ref_f16, triton_bwd(A, B, dC), atol=1e-2)
print("Triton fwd:", "\u2705" if ok_fwd else "\u274c", " bwd:", "\u2705" if ok_bwd else "\u274c")


## TileLang Implementation

In [ ]:
BN, BM = 64, 64

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_fwd(A, B, BLOCK_N: int, BLOCK_M: int):
    N_, M_ = T.const("N, M")
    dtype = T.float16
    A: T.Tensor((N_, M_), dtype);  B: T.Tensor((M_,), dtype)
    C = T.empty((N_, M_), dtype)
    with T.Kernel(N_ // BLOCK_N, M_ // BLOCK_M, threads=256) as (pn, pm):
        for i, j in T.Parallel(BLOCK_N, BLOCK_M):
            r, c = pn * BLOCK_N + i, pm * BLOCK_M + j
            val = A[r, c] * B[c]
            C[r, c] = T.if_then_else(val > dtype(0), val, dtype(0))
    return C

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_bwd(A, B, dC, BLOCK_N: int, BLOCK_M: int):
    N_, M_ = T.const("N, M")
    dtype = T.float16
    A:  T.Tensor((N_, M_), dtype);  B: T.Tensor((M_,), dtype)
    dC: T.Tensor((N_, M_), dtype)
    dA = T.empty((N_, M_), dtype)
    with T.Kernel(N_ // BLOCK_N, M_ // BLOCK_M, threads=256) as (pn, pm):
        for i, j in T.Parallel(BLOCK_N, BLOCK_M):
            r, c = pn * BLOCK_N + i, pm * BLOCK_M + j
            gate = T.if_then_else(A[r, c] * B[c] > dtype(0), dtype(1), dtype(0))
            dA[r, c] = dC[r, c] * B[c] * gate
    return dA

hyp = {"N": N, "M": M, "BLOCK_N": BN, "BLOCK_M": BM}
k_fwd = tl_fwd.compile(**hyp)
k_bwd = tl_bwd.compile(**hyp)

ok_f = torch.allclose(C_ref,      k_fwd(A.clone(), B.clone()),             atol=1e-2)
ok_b = torch.allclose(dA_ref_f16, k_bwd(A.clone(), B.clone(), dC.clone()), atol=1e-2)
print("TileLang fwd:", "\u2705" if ok_f else "\u274c", " bwd:", "\u2705" if ok_b else "\u274c")


## Performance Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

fwd_ms = [bench(ref_fwd,    [A, B]),
          bench(triton_fwd, [A, B]),
          bench(k_fwd,      [A, B])]
bwd_ms = [bench(ref_bwd,    [A, B, dC]),
          bench(triton_bwd, [A, B, dC]),
          bench(k_bwd,      [A, B, dC])]

labels = ["PyTorch", "Triton", "TileLang"]
colors = ["#4C72B0", "#55A868", "#C44E52"]

print("Forward: ", {l: f"{ms:.4f} ms" for l, ms in zip(labels, fwd_ms)})
print("Backward:", {l: f"{ms:.4f} ms" for l, ms in zip(labels, bwd_ms)})

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
for ax, data, title in zip(axes, [fwd_ms, bwd_ms],
    ["Forward (Broadcast Mul+ReLU)", "Backward (gradient of A)"]):
    bars = ax.bar(labels, data, color=colors, width=0.5, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, data):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(data)*0.01,
                f"{val:.4f} ms", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Latency (ms)")
    ax.set_title(f"{title}\n(N={N}, M={M}, float16)")
    ax.set_ylim(0, max(data)*1.3)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()